# LLM run inspection

Run one rendered scenario transition through an LLM, log the result, and inspect the submitted move.

Set at least
* `SCENARIO_NAME` to the name of a file in `outputs/scenarios`(without .json suffix!)
* `TRANSITION_INDEX` to the explicit transition from the selected scenario
* `MODEL_NAME` to the name of the model (options specified in `config/model_configs.yaml`)


The move is shown in one 2D plot per axis pair containing the move axis. A 4D move on axis 0 therefore produces the planes (0, 1), (0, 2), and (0, 3).

In [1]:
from visualization.run_figures import (
    call_prepared_llm_transition,
    display_llm_prompt,
    display_llm_response,
    display_llm_run_summary,
    finalize_llm_transition,
    plot_llm_run_move,
    prepare_llm_transition,
)

In [2]:
SCENARIO_NAME = "generator_3d"
TRANSITION_INDEX = 9
MODEL_NAME = "openrouter_gpt-5"
REASONING_EFFORT = "low"

In [3]:
# Local preparation only: load scenario, reconstruct board, and build prompts.
prepared = prepare_llm_transition(
    scenario_name=SCENARIO_NAME,
    transition_index=TRANSITION_INDEX,
    model_name=MODEL_NAME,
    reasoning_effort=REASONING_EFFORT,
)
display_llm_prompt(prepared)


### Prompt sources

- System template: [`prompts/system.txt`](../prompts/system.txt)
- User template: [`prompts/user.txt`](../prompts/user.txt)
- Model config: [`config/model_configs.yaml`](../config/model_configs.yaml)
- Scenario: [`generator_3d.json`](../outputs/scenarios/generator_3d.json)
- Backend: `litellm`

### System prompt

```text
You solve a multidimensional formal-language Scrabble benchmark.
Follow the supplied rules exactly and return one move in the required structured format.
Do not use tools or add explanations.
```

### User prompt

```text
Place exactly one contiguous sequence. Maximize the number of newly placed rack symbols.

## Move geometry
- Coordinates are zero-based vectors with 3 entries.
- `start` is the first coordinate. `axis` advances one coordinate component per symbol.
- Axis selects the board dimension along which the sequence advances: axis 0 advances coordinate index 0, axis 1 advances coordinate index 1, axis 2 advances coordinate index 2.

## Validity rules
- The submitted sequence must be accepted by the formal language.
- Existing cells may be reused only with their existing symbol.
- Reuse at least one existing cell and place at least one new symbol.
- Only newly placed symbols consume the rack, including multiplicities.
- Do not reuse a cell whose existing word already runs along the chosen axis.
- The cell immediately before and after the submitted sequence on its axis must not continue an existing word.
- A newly placed cell must not extend an already-valid word on any perpendicular axis.
- After placement, every maximal contiguous line of length greater than one that touches the move, on every axis, must be accepted by the formal language.

Formal language:
Language ID: generator_3d_grammar
Alphabet: {A, J, R, T, Z}
k: 3
Minimum word length: 3
Forbidden snippets: {A A J, A A R, A A Z, A J A, A R R, A R T, A R Z, A Z J, A Z R, J J A, J R A, J R J, J R R, J T Z, R A A, R J J, R J R, R R Z, R T A, T J Z, T R J, T R T, T T R, T T Z, T Z T, Z A A, Z A T, Z J J, Z J Z, Z Z J}
A sequence is valid iff it has the minimum length of 3 and contains no forbidden snippet.

Board configuration:
[omitted from notebook display: 46 occupied cells]

Rack:
["J", "J", "J", "T", "T", "Z"]

```


In [4]:
timed_response = call_prepared_llm_transition(prepared)

In [5]:
context = finalize_llm_transition(prepared, timed_response)
display_llm_response(timed_response)
display_llm_run_summary(context)

sequence,OK
spatial,OK
overlap,OK
no word extension,OK
cross words,OK
rack,OK


In [6]:
for figure in plot_llm_run_move(context, move_source="parsed"):
    display(figure)

# Keep in Mind

Ground truth != only solution

In [7]:
for figure in plot_llm_run_move(context, move_source="ground_truth"):
    display(figure)